# Gold Customer 360 & Support Analytics

Builds gold-layer analytical tables from silver data: customer 360 summary, product performance, enriched order summary, support ticket analytics, and daily sales trends.

In [0]:
%run "/Users/jaygupta9584@gmail.com/Intelligent Customer 360 & Support Analytics Platform/Common/CommonHelper"

In [0]:
# --- Gold: Star Schema (Dimensions + Fact Sales) ---
# Read silver once, build dimension tables, then fact_sales at order-item grain.

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read silver tables once
orders_s = spark.table("customer_360.silver.orders")
order_items_s = spark.table("customer_360.silver.order_items")
products_s = spark.table("customer_360.silver.products")
payments_s = spark.table("customer_360.silver.payments")
customers_s = spark.table("customer_360.silver.customer")
support_s = spark.table("customer_360.silver.support_tickets")

# ── Dimension Tables ──
dim_customer = customers_s.select(
    "customer_id", "customer_name", "email", "phone",
    "city", "state", "country", "customer_segment", "signup_date")

dim_customer.write.mode("overwrite").saveAsTable("customer_360.gold.dim_customer")

dim_product = products_s.select(
    "product_id", "product_name", "category", "sub_category", "brand", "price", "cost")

dim_product.write.mode("overwrite").saveAsTable("customer_360.gold.dim_product")

dim_order = orders_s.select(
    "order_id", "customer_id", "order_date", "order_status",
    "total_amount", "shipping_city", "shipping_state", "created_at", "updated_at")

dim_order.write.mode("overwrite").saveAsTable("customer_360.gold.dim_order")

dim_payment = payments_s.select(
    "payment_id", "order_id", "customer_id", "payment_date",
    "payment_method", "payment_status", "amount")

dim_payment.write.mode("overwrite").saveAsTable("customer_360.gold.dim_payment")

dim_date = (
    orders_s.select(F.col("order_date").cast("date").alias("dt")).distinct()
    .select(
        F.col("dt").alias("date"),
        F.year("dt").alias("year"), F.quarter("dt").alias("quarter"),
        F.month("dt").alias("month"), F.date_format("dt", "MMMM").alias("month_name"),
        F.dayofweek("dt").alias("day_of_week"), F.date_format("dt", "EEEE").alias("day_name"),
        F.weekofyear("dt").alias("week_of_year"),
        F.when(F.dayofweek("dt").isin(1, 7), True).otherwise(False).alias("is_weekend"),
    ).orderBy("date"))

dim_date.write.mode("overwrite").saveAsTable("customer_360.gold.dim_date")

# ── Fact Sales (order-item grain) ──
payment_first = (
    payments_s
    .withColumn("rn", F.row_number().over(Window.partitionBy("order_id").orderBy("payment_date")))
    .filter(F.col("rn") == 1).drop("rn"))

ticket_counts = (
    support_s.filter(F.col("order_id").isNotNull())
    .groupBy("order_id").agg(F.count("*").alias("support_ticket_count")))

fact_sales = (
    order_items_s.alias("oi")
    .join(orders_s.alias("o"), F.col("oi.order_id") == F.col("o.order_id"), "left")
    .join(products_s.alias("p"), F.col("oi.product_id") == F.col("p.product_id"), "left")
    .join(customers_s.alias("c"), F.col("o.customer_id") == F.col("c.customer_id"), "left")
    .join(payment_first.alias("pm"), F.col("o.order_id") == F.col("pm.order_id"), "left")
    .join(ticket_counts.alias("tc"), F.col("o.order_id") == F.col("tc.order_id"), "left")
    .select(
        F.col("oi.order_item_id"), F.col("oi.order_id"), F.col("o.customer_id"),
        F.col("oi.product_id"), F.col("pm.payment_id"),
        F.col("o.order_date"), F.col("o.order_status"), F.col("c.customer_segment"),
        F.col("p.category"), F.col("p.sub_category"), F.col("p.brand"),
        F.col("pm.payment_method"), F.col("pm.payment_status"),
        F.col("o.shipping_city"), F.col("o.shipping_state"),
        F.col("oi.quantity"), F.col("oi.unit_price"), F.col("oi.discount"),
        (F.col("oi.quantity") * F.col("oi.unit_price")).alias("gross_amount"),
        (F.col("oi.quantity") * F.col("oi.unit_price") - F.col("oi.discount")).alias("net_amount"),
        F.col("p.cost").alias("product_cost"),
        (F.col("oi.quantity") * F.col("oi.unit_price") - F.col("oi.discount")
         - F.col("oi.quantity") * F.col("p.cost")).alias("profit"),
        F.coalesce(F.col("tc.support_ticket_count"), F.lit(0)).alias("support_ticket_count"),
    )
)

fact_sales.write.mode("overwrite").saveAsTable("customer_360.gold.fact_sales")

print("--- Star Schema ---")
for t in ["dim_customer", "dim_product", "dim_order", "dim_payment", "dim_date", "fact_sales"]:
    print(f"  {t}: {spark.table(f'customer_360.gold.{t}').count():,} rows")

In [0]:
# --- Gold: Fact Support (ticket grain) ---
# Reuses silver DataFrames from star schema cell.
# One row per support ticket with additive measures and dimension FKs.

from pyspark.sql import functions as F

fact_support = (
    support_s.alias("t")
    .join(customers_s.alias("c"), F.col("t.customer_id") == F.col("c.customer_id"), "left")
    .join(orders_s.alias("o"), F.col("t.order_id") == F.col("o.order_id"), "left")
    .withColumn("resolution_hours",
        F.when(F.col("t.resolved_at").isNotNull(),
            F.round((F.unix_timestamp(F.col("t.resolved_at"))
                     - F.unix_timestamp(F.col("t.created_at"))) / 3600.0, 2)
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn("sla_met",
        F.when(
            (F.col("t.resolved_at").isNotNull())
            & (F.col("resolution_hours") <= 48), 1
        ).otherwise(0)
    )
    .select(
        # --- Keys ---
        F.col("t.ticket_id"),
        F.col("t.customer_id"),
        F.col("t.order_id"),
        # --- Degenerate dimensions ---
        F.col("t.issue_category"),
        F.col("t.priority"),
        F.col("t.status"),
        F.col("c.customer_segment"),
        # --- Temporal context ---
        F.col("t.created_at").alias("ticket_created_at"),
        F.col("t.resolved_at"),
        F.col("t.updated_at"),
        # --- Additive measures ---
        F.col("resolution_hours"),
        F.when(F.col("t.status") == "RESOLVED", 1).otherwise(0).alias("is_resolved"),
        F.when(F.col("t.status").isin("OPEN", "IN_PROGRESS"), 1).otherwise(0).alias("is_open"),
        F.when(F.col("t.order_id").isNotNull(), 1).otherwise(0).alias("has_order"),
        F.when(F.col("t.priority") == "URGENT", 4)
         .when(F.col("t.priority") == "HIGH", 3)
         .when(F.col("t.priority") == "MEDIUM", 2)
         .when(F.col("t.priority") == "LOW", 1)
         .otherwise(0).alias("priority_rank"),
        F.col("sla_met"),
    )
)

fact_support.write.mode("overwrite").saveAsTable("customer_360.gold.fact_support")

print(f"  fact_support: {spark.table('customer_360.gold.fact_support').count():,} rows")

In [0]:
# --- Gold: Aggregate Analytical Tables ---
# Independent analytical views built from silver via SQL CTEs.

# 1. Customer 360 Summary — one row per customer
spark.sql("""
CREATE OR REPLACE TABLE customer_360.gold.customer_summary AS
WITH order_metrics AS (
    SELECT customer_id, COUNT(*) AS total_orders,
        COALESCE(SUM(total_amount), 0) AS total_spend,
        COALESCE(AVG(total_amount), 0) AS avg_order_value,
        MIN(order_date) AS first_order_date, 
        MAX(order_date) AS last_order_date,
        SUM(CASE WHEN order_status = 'DELIVERED' THEN 1 ELSE 0 END) AS delivered_orders,
        SUM(CASE WHEN order_status = 'PROCESSING' THEN 1 ELSE 0 END) AS processing_orders,
        SUM(CASE WHEN order_status = 'SHIPPED' THEN 1 ELSE 0 END) AS shipped_orders,
        SUM(CASE WHEN order_status = 'RETURNED' THEN 1 ELSE 0 END) AS returned_orders,
        SUM(CASE WHEN order_status = 'CANCELLED' THEN 1 ELSE 0 END) AS cancelled_orders
    FROM customer_360.silver.orders GROUP BY customer_id
),

payment_metrics AS (
    SELECT customer_id,
        COALESCE(SUM(CASE WHEN payment_status = 'SUCCESS' THEN amount ELSE 0 END), 0) AS total_paid,
        ROUND(COALESCE(SUM(CASE WHEN payment_status = 'SUCCESS' THEN 1 ELSE 0 END) * 100.0 / NULLIF(COUNT(*), 0), 0), 2) AS payment_success_rate
    FROM customer_360.silver.payments GROUP BY customer_id
),

preferred_payment AS (
    SELECT customer_id, payment_method AS preferred_payment_method FROM (
        SELECT customer_id, payment_method,
            ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY COUNT(*) DESC) AS rn
        FROM customer_360.silver.payments GROUP BY customer_id, payment_method
    ) WHERE rn = 1
),
support_metrics AS (
    SELECT customer_id, COUNT(*) AS total_tickets,
        SUM(CASE WHEN status IN ('OPEN', 'IN_PROGRESS') THEN 1 ELSE 0 END) AS open_tickets,
        SUM(CASE WHEN status = 'RESOLVED' THEN 1 ELSE 0 END) AS resolved_tickets,
        ROUND(AVG(CASE WHEN resolved_at IS NOT NULL
            THEN (UNIX_TIMESTAMP(resolved_at) - UNIX_TIMESTAMP(created_at)) / 3600.0 END), 2) AS avg_resolution_hours
    FROM customer_360.silver.support_tickets GROUP BY customer_id
),
event_metrics AS (
    SELECT 
        customer_id, 
        COUNT(*) AS total_events, 
        MAX(event_timestamp) AS last_event_date
    FROM customer_360.silver.customer_events GROUP BY customer_id
),
preferred_device AS (
    SELECT customer_id, device_type AS preferred_device FROM (
        SELECT customer_id, device_type,
            ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY COUNT(*) DESC) AS rn
        FROM customer_360.silver.customer_events GROUP BY customer_id, device_type
    ) WHERE rn = 1
),
most_viewed_page AS (
    SELECT customer_id, page AS most_viewed_page FROM (
        SELECT customer_id, page,
            ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY COUNT(*) DESC) AS rn
        FROM customer_360.silver.customer_events GROUP BY customer_id, page
    ) WHERE rn = 1
)
SELECT c.customer_id, c.customer_name, c.email, c.phone, c.city, c.state, c.country,
    c.customer_segment, c.signup_date,
    COALESCE(o.total_orders, 0) AS total_orders, COALESCE(o.total_spend, 0) AS total_spend,
    COALESCE(o.avg_order_value, 0) AS avg_order_value, o.first_order_date, o.last_order_date,
    COALESCE(o.delivered_orders, 0) AS delivered_orders, COALESCE(o.processing_orders, 0) AS processing_orders,
    COALESCE(o.shipped_orders, 0) AS shipped_orders, COALESCE(o.returned_orders, 0) AS returned_orders,
    COALESCE(o.cancelled_orders, 0) AS cancelled_orders,
    COALESCE(p.total_paid, 0) AS total_paid, COALESCE(p.payment_success_rate, 0) AS payment_success_rate,
    pp.preferred_payment_method,
    COALESCE(s.total_tickets, 0) AS total_tickets, COALESCE(s.open_tickets, 0) AS open_tickets,
    COALESCE(s.resolved_tickets, 0) AS resolved_tickets, s.avg_resolution_hours,
    COALESCE(e.total_events, 0) AS total_events, e.last_event_date,
    d.preferred_device, mvp.most_viewed_page
FROM customer_360.silver.customer c
LEFT JOIN order_metrics o ON c.customer_id = o.customer_id
LEFT JOIN payment_metrics p ON c.customer_id = p.customer_id
LEFT JOIN preferred_payment pp ON c.customer_id = pp.customer_id
LEFT JOIN support_metrics s ON c.customer_id = s.customer_id
LEFT JOIN event_metrics e ON c.customer_id = e.customer_id
LEFT JOIN preferred_device d ON c.customer_id = d.customer_id
LEFT JOIN most_viewed_page mvp ON c.customer_id = mvp.customer_id
""")

# 2. Product Performance — one row per product
spark.sql("""
CREATE OR REPLACE TABLE customer_360.gold.product_performance AS
WITH item_details AS (
    SELECT oi.product_id, oi.order_id, oi.quantity, oi.unit_price, oi.discount,
        (oi.quantity * oi.unit_price - oi.discount) AS gross_revenue, p.cost
    FROM customer_360.silver.order_items oi
    JOIN customer_360.silver.products p ON oi.product_id = p.product_id
), order_customers AS (
    SELECT DISTINCT order_id, customer_id FROM customer_360.silver.orders
)
SELECT p.product_id, p.product_name, p.category, p.sub_category, p.brand, p.price, p.cost,
    ROUND(p.cost / NULLIF(p.price, 0) * 100, 2) AS cost_ratio_pct,
    COALESCE(SUM(id.quantity), 0) AS total_units_sold,
    COALESCE(SUM(id.gross_revenue), 0) AS total_revenue,
    COALESCE(SUM(id.gross_revenue - id.quantity * id.cost), 0) AS total_profit,
    COUNT(DISTINCT id.order_id) AS order_count,
    COUNT(DISTINCT oc.customer_id) AS unique_customers,
    COALESCE(ROUND(AVG(id.discount), 2), 0) AS avg_discount
FROM customer_360.silver.products p
LEFT JOIN item_details id ON p.product_id = id.product_id
LEFT JOIN order_customers oc ON id.order_id = oc.order_id
GROUP BY p.product_id, p.product_name, p.category, p.sub_category, p.brand, p.price, p.cost
""")

# 3. Order Summary — enriched orders
spark.sql("""
CREATE OR REPLACE TABLE customer_360.gold.order_summary AS
SELECT o.order_id, o.customer_id, c.customer_name, c.customer_segment,
    o.order_date, o.order_status, o.total_amount, o.shipping_city, o.shipping_state,
    pm.payment_method, pm.payment_status, pm.payment_date, pm.amount AS payment_amount,
    COALESCE(oi.item_count, 0) AS item_count, COALESCE(oi.total_quantity, 0) AS total_quantity,
    COALESCE(oi.total_item_value, 0) AS total_item_value, COALESCE(oi.total_discount, 0) AS total_discount,
    COALESCE(st.ticket_count, 0) AS support_ticket_count
FROM customer_360.silver.orders o
LEFT JOIN customer_360.silver.customer c ON o.customer_id = c.customer_id
LEFT JOIN (
    SELECT order_id, payment_method, payment_status, payment_date, amount,
        ROW_NUMBER() OVER(PARTITION BY order_id ORDER BY payment_date) AS rn
    FROM customer_360.silver.payments
) pm ON o.order_id = pm.order_id AND pm.rn = 1
LEFT JOIN (
    SELECT order_id, COUNT(*) AS item_count, SUM(quantity) AS total_quantity,
        SUM(quantity * unit_price) AS total_item_value, SUM(discount) AS total_discount
    FROM customer_360.silver.order_items GROUP BY order_id
) oi ON o.order_id = oi.order_id
LEFT JOIN (
    SELECT order_id, COUNT(*) AS ticket_count
    FROM customer_360.silver.support_tickets WHERE order_id IS NOT NULL GROUP BY order_id
) st ON o.order_id = st.order_id
""")

# 4. Support Analytics — ticket-level with SLA
spark.sql("""
CREATE OR REPLACE TABLE customer_360.gold.support_analytics AS
SELECT t.ticket_id, t.customer_id, c.customer_name, c.customer_segment,
    c.city AS customer_city, c.state AS customer_state,
    t.order_id, o.order_date, o.total_amount AS order_amount,
    t.issue_category, t.subject, t.priority, t.status, t.resolution,
    t.created_at AS ticket_created_at, t.resolved_at, t.updated_at,
    CASE WHEN t.resolved_at IS NOT NULL
        THEN ROUND((UNIX_TIMESTAMP(t.resolved_at) - UNIX_TIMESTAMP(t.created_at)) / 3600.0, 2)
        ELSE NULL END AS resolution_hours,
    CASE WHEN t.resolved_at IS NOT NULL
            AND (UNIX_TIMESTAMP(t.resolved_at) - UNIX_TIMESTAMP(t.created_at)) / 3600.0 <= 48
        THEN 'MET' WHEN t.resolved_at IS NOT NULL THEN 'BREACHED' ELSE 'PENDING' END AS sla_status,
    CASE WHEN t.priority = 'URGENT' THEN 4 WHEN t.priority = 'HIGH' THEN 3
         WHEN t.priority = 'MEDIUM' THEN 2 WHEN t.priority = 'LOW' THEN 1 ELSE 0 END AS priority_rank
FROM customer_360.silver.support_tickets t
LEFT JOIN customer_360.silver.customer c ON t.customer_id = c.customer_id
LEFT JOIN customer_360.silver.orders o ON t.order_id = o.order_id
""")

# 5. Daily Sales — daily aggregation
spark.sql("""
CREATE OR REPLACE TABLE customer_360.gold.daily_sales AS
SELECT DATE(o.order_date) AS sales_date, COUNT(*) AS total_orders,
    COALESCE(SUM(o.total_amount), 0) AS total_revenue,
    COALESCE(AVG(o.total_amount), 0) AS avg_order_value,
    COALESCE(SUM(oi.total_quantity), 0) AS total_items_sold,
    COUNT(DISTINCT o.customer_id) AS unique_customers,
    SUM(CASE WHEN o.order_status = 'DELIVERED' THEN 1 ELSE 0 END) AS delivered_orders,
    SUM(CASE WHEN o.order_status = 'CANCELLED' THEN 1 ELSE 0 END) AS cancelled_orders,
    SUM(CASE WHEN o.order_status = 'RETURNED' THEN 1 ELSE 0 END) AS returned_orders
FROM customer_360.silver.orders o
LEFT JOIN (SELECT order_id, SUM(quantity) AS total_quantity FROM customer_360.silver.order_items GROUP BY order_id) oi
    ON o.order_id = oi.order_id
GROUP BY DATE(o.order_date) ORDER BY sales_date
""")

print("--- Aggregate Tables ---")
for t in ["customer_summary", "product_performance", "order_summary", "support_analytics", "daily_sales"]:
    print(f"  {t}: {spark.table(f'customer_360.gold.{t}').count():,} rows")

In [0]:
# --- Gold Layer Summary ---
# print("=" * 55)
# print("Gold Layer: customer_360.gold")
# print("=" * 55)
# gold_tables = [
#     "dim_customer", "dim_product", "dim_order", "dim_payment", "dim_date",
#     "fact_sales", "fact_support", "customer_summary", "product_performance",
#     "order_summary", "support_analytics", "daily_sales",
# ]
# for t in gold_tables:
#     n = spark.table(f"customer_360.gold.{t}").count()
    # print(f"  {t:25s} {n:>10,} rows")
# print("=" * 55)
# display(spark.table("customer_360.gold.fact_sales").limit(5))
# display(spark.table("customer_360.gold.fact_support").limit(5))
# display(spark.table("customer_360.gold.customer_summary").limit(5))